# 00 · Datos, pesos y reproducibilidad

**Proyecto:** SAM sobre Sentinel-1 para segmentación de agua (Sen1Floods11). INF659, 2026-2.

Este notebook descarga y verifica todo lo que usan los demás:

1. Pesos de `facebook/sam-vit-base` en una **revisión inmutable** (commit fijado) y en formato `safetensors`, con el SHA-256 contrastado contra el publicado por Hugging Face.
2. Chips etiquetados a mano de Sen1Floods11 v1.1, particiones `valid` y `train`, desde el bucket oficial. **`test` y `bolivia` no se descargan**: quedan reservados para la evaluación final (E3).
3. Comprobación de que los 20 archivos del E1 (manifiesto del Anexo B) coinciden bit a bit con los descargados.
4. Aplicación del criterio de exclusión declarado (más de 95 % de píxeles sin etiqueta) y la subpartición `train-dev` para el barrido de hiperparámetros de H2.

Las descargas son secuenciales y con velocidad limitada (2.5 MB/s). Si los archivos ya existen, no se vuelven a bajar.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
from inf659sam import config, data
from inf659sam.cache import save_json
from inf659sam.experiments import evaluable_chips

config.setup_runtime(seed=0)
print(json.dumps(config.environment_report(), indent=1))
print("Datos  :", config.DATA_DIR)
print("Modelo :", config.MODEL_DIR)
print("Cache  :", config.CACHE_DIR)

Entorno: {'seed': 0, 'torch_threads': 2, 'priority': 'baja'}


{
 "python": "3.14.7",
 "platform": "Windows-11-10.0.26200-SP0",
 "cpu": "AMD64 Family 25 Model 33 Stepping 2, AuthenticAMD",
 "ram_gb": 31.9,
 "torch": "2.14.1+cpu",
 "transformers": "5.19.0",
 "numpy": "2.5.3",
 "scipy": "1.18.1",
 "skimage": "0.26.0",
 "device": "cpu",
 "torch_threads": 2,
 "model_id": "facebook/sam-vit-base",
 "model_revision": "70c1a07f894ebb5b307fd9eaaee97b9dfc16068f"
}
Datos  : <INF659_ENV>\data
Modelo : <INF659_ENV>\models\sam-vit-base
Cache  : <INF659_ENV>\cache


## 1. Pesos de SAM (revisión fijada)

In [2]:
model_dir = data.ensure_model()
pesos = {f: data.sha256(model_dir / f) for f in config.MODEL_FILES}
for f, h in pesos.items():
    print(f"{f:28s} {h}")
print("Revisión:", config.MODEL_REVISION)
print("SHA-256 de model.safetensors coincide con Hugging Face:", pesos["model.safetensors"] == data.MODEL_SAFETENSORS_SHA256)

config.json                  5ebd0d8643b486f3a716bf17c2a15531eb818b2b96ff0c6c5dcc88fa015161af
preprocessor_config.json     225545a743c654e3c495ec6f545a0eaba57c8ba3fbbd8483b3cb1c0fc58db517
model.safetensors            892c410e496344e527255ccdcb2cb7244a609acb5389c7c4fdba1288f861c579
Revisión: 70c1a07f894ebb5b307fd9eaaee97b9dfc16068f
SHA-256 de model.safetensors coincide con Hugging Face: True


## 2. Particiones oficiales y descarga

In [3]:
import pandas as pd

splits = {s: data.read_split(s) for s in ("train", "valid", "test", "bolivia")}
tabla = pd.DataFrame({s: pd.Series([data.region_of(c) for c in v]).value_counts() for s, v in splits.items()}).fillna(0).astype(int)
tabla.loc["TOTAL"] = tabla.sum()
tabla

,train,valid,test,bolivia
Bolivia,0,0,0,15
Ghana,31,11,11,0
India,40,14,14,0
Mekong,18,6,6,0
Nigeria,10,4,4,0
Pakistan,16,6,6,0
Paraguay,39,14,14,0
Somalia,15,5,6,0
Spain,18,6,6,0
Sri-Lanka,24,9,9,0


In [4]:
hashes = {}
for s in ("valid", "train"):
    print(f"Descargando/verificando {s} ({len(splits[s])} chips)")
    hashes.update(data.ensure_chips(splits[s], log_every=50))
data.write_manifest(config.RESULTS_DIR / "manifiesto_datos.json",
                    {"valid": splits["valid"], "train": splits["train"]}, hashes,
                    extra={"modelo": {"id": config.MODEL_ID, "revision": config.MODEL_REVISION, "sha256": pesos},
                           "no_descargado": {"test": len(splits["test"]), "bolivia": len(splits["bolivia"])}})
print("Archivos verificados:", len(hashes))

Descargando/verificando valid (89 chips)
  chips listos: 50/89


  chips listos: 89/89


Descargando/verificando train (252 chips)
  chips listos: 50/252


  chips listos: 100/252


  chips listos: 150/252


  chips listos: 200/252


  chips listos: 250/252


  chips listos: 252/252


Archivos verificados: 682


## 3. Los 20 archivos del E1 coinciden con los oficiales

Hashes copiados del Anexo B del documento E1 (manifiesto del equipo).

In [5]:
SHA_E1 = {
    "Ghana_5079_LabelHand.tif": "042647898ddeb5be7fd46494c5bc413e22413f21465b90d0d13f57d16f23ffa0",
    "Ghana_5079_S1Hand.tif": "5a598a0050e522fc8faf3ffbe8ce4edc3ad642d16c18ad1faa54574adf76f3c7",
    "India_1050276_LabelHand.tif": "b0f21694aa0a1e5a48ccc4f66e12f5a8b3f16e606164e6cafd789b6aed121ba9",
    "India_1050276_S1Hand.tif": "14a6d3889427e4b3e4a2b13fbc664cbc9bc84c33a3f2a63553b67cc0743bb73f",
    "Mekong_1149855_LabelHand.tif": "e90aaedbb51c617c6250c06cbeff13a14c992c46e5df1fe5b8988a5b239308b6",
    "Mekong_1149855_S1Hand.tif": "e53925ce5265c89099b9d3aa1915661173ffd34375c751e2e24c1b1a4bd07587",
    "Nigeria_31096_LabelHand.tif": "c8da655114d488f6097de39492da9e27f065feef58764c70ef5d7b8683ae8542",
    "Nigeria_31096_S1Hand.tif": "928f806af7f3713ffce13c152f09713ab25198b646902ea0fe4e780a8ab70350",
    "Pakistan_43105_LabelHand.tif": "60fe4edb4e6ce61fad7cd3e8810e0f030712fccf2053e7996c786941ffcb0130",
    "Pakistan_43105_S1Hand.tif": "1594a4ab57da8f3ef16816661bdbba94669236364605465832047a8560c2252c",
    "Paraguay_305760_LabelHand.tif": "fcb5f3b21ae74bd8275226d843d6254a982448b7669f43dbff6fea44647db24a",
    "Paraguay_305760_S1Hand.tif": "ff60d990884d9ac12ab66c6604a4a9b33d7829e9a9a1d332142cdd048ab042f8",
    "Somalia_12849_LabelHand.tif": "9b7cb1c8a378c193b1be9405c6ea200c38c3fc80300d190184c0226383c0afcb",
    "Somalia_12849_S1Hand.tif": "da7ae40306d1615d3bd11627cfdf7086b6f3a7335c709a69a146e725e25484ee",
    "Spain_1199913_LabelHand.tif": "27fedadf6819805196eb1061a21e11ccb7ba26648a2424917b9de3174d45f2e0",
    "Spain_1199913_S1Hand.tif": "343e99bb775af47d2fb690d04c1c71c56c751f179c13aa56c84992fd4e6b740f",
    "Sri-Lanka_85652_LabelHand.tif": "101a849cbddaa60858d1ef42bab6c88d762c5b4aad31ddb0a4b8f0d9bfedb842",
    "Sri-Lanka_85652_S1Hand.tif": "401dedb1de802b0793ada0e2f7288b2bdf2ccd1abff647be9be0b96948229210",
    "USA_826217_LabelHand.tif": "f2a6c8a54d0fe0e6f36ee7f8930f11c8c1190f11873e97275d33e3d89a81ba6d",
    "USA_826217_S1Hand.tif": "7d39bd4c52057b00a368342756883e32ef288fcc533fbe1775a9357a9df3b205",
}
coinciden = {k: hashes.get(k) == v for k, v in SHA_E1.items()}
print(f"Coinciden {sum(coinciden.values())} de {len(SHA_E1)} archivos del manifiesto del E1")
assert all(coinciden.values())
print("Regla de selección del E1 (primera fila por región en valid):", data.first_chip_per_region(splits["valid"]))

Coinciden 20 de 20 archivos del manifiesto del E1
Regla de selección del E1 (primera fila por región en valid): ['Ghana_5079', 'India_1050276', 'Mekong_1149855', 'Nigeria_31096', 'Pakistan_43105', 'Paraguay_305760', 'Somalia_12849', 'Spain_1199913', 'Sri-Lanka_85652', 'USA_826217']


## 4. Criterio de exclusión y subpartición `train-dev`

In [6]:
keep_v, log_v = evaluable_chips(splits["valid"])
keep_t, log_t = evaluable_chips(splits["train"])
core, dev = data.stratified_holdout(keep_t, frac=0.2, seed=0)
print(f"valid: {len(keep_v)} evaluables de {len(splits['valid'])}; excluidos {[r['chip'] for r in log_v if r['excluido']]}")
print(f"train: {len(keep_t)} evaluables de {len(splits['train'])}; excluidos {[r['chip'] for r in log_t if r['excluido']]}")
print(f"train-core: {len(core)}   train-dev: {len(dev)}")
save_json({"valid": keep_v, "train": keep_t, "train_core": core, "train_dev": dev,
           "exclusion_valid": log_v, "exclusion_train": log_t,
           "criterio": f"frac_nodata_label > {config.NODATA_EXCLUSION_THRESHOLD}"},
          config.RESULTS_DIR / "particiones_usadas.json")

valid: 86 evaluables de 89; excluidos ['Ghana_5079', 'Ghana_234935', 'Ghana_277']
train: 250 evaluables de 252; excluidos ['Ghana_161233', 'Ghana_26376']
train-core: 199   train-dev: 51
